# Ground-Truth Driver / Non-Driver Gene Retrieval (Unmatched Negatives)

Retrieves the ground-truth driver and non-driver gene sets (protein-coding universe -> driver
exclusion -> mutation-frequency filter -> disease-pathway filter) and saves **four separate
files**:

1. `driver_genes.txt` -- one gene per line, the ground-truth positive (driver) set
2. `non_driver_genes.txt` -- one gene per line, the ground-truth negative (non-driver) set
3. `unlabeled_genes.txt` -- one gene per line, protein-coding genes that ended up with
   neither label (single-source driver candidates plus genes dropped by the
   genes dropped by the mutation-frequency/pathway filters)
4. `gene_labels.csv` -- drivers + non-drivers combined into one labeled table (`label`
   column: 1 = driver, 0 = non-driver; unlabeled genes are not included in this file)

Non-driver genes are **not** length/expression-matched here -- whatever survives the
mutation-frequency and disease-pathway filters is kept as-is as the negative class.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import config
from src.driver_labeling import (
    apply_mutation_frequency_filter,
    apply_pathway_filter,
    build_gene_labels,
)
from src.gene_universe import build_protein_coding_gene_universe
from collections import Counter
from src.negative_sampling import (
    load_bailey_genes,
    load_cgc_genes,
    load_intogen_genes,
    load_ncg_genes,
)
from src.pathway_filter import build_disease_pathway_genes

config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] USE_MUTATION_FILTER: {config.USE_MUTATION_FILTER}")
print(f"[CONFIG] USE_PATHWAY_FILTER: {config.USE_PATHWAY_FILTER}")


[CONFIG] USE_MUTATION_FILTER: True
[CONFIG] USE_PATHWAY_FILTER: True


## 1. Protein-coding gene universe

Parses the GENCODE GTF and keeps only `gene_type == "protein_coding"`.


In [2]:
all_genes = build_protein_coding_gene_universe(
    "../../data/gencode.v49.basic.annotation.gtf"
)
# Use a single symbol convention for GENCODE, CGC, TCGA, and Reactome joins.
all_genes = {gene.upper() for gene in all_genes}
pd.DataFrame(sorted(all_genes), columns=["gene_name"]).to_csv(
    config.GENCODE_GENES_FILE, index=False
)
print(f"[GENE UNIVERSE] {len(all_genes)} protein-coding genes")


[GENE UNIVERSE] 20070 protein-coding genes


## 2. Ground-truth driver genes and the expanded exclusion set

All four driver references (NCG, CGC, IntOGen, and Bailey et al. 2018) are used. A gene
receives a positive label only with support from at least two sources; the complete union
is still excluded when deriving non-driver candidates. This keeps single-source candidates
unlabeled rather than incorrectly treating them as negatives.


In [3]:
MIN_DRIVER_SOURCE_SUPPORT = 2
reference_sources = {
    "NCG": load_ncg_genes("../data/NCG_cancergenes.tsv"),
    "CGC": load_cgc_genes(config.CGC_CENSUS_FILE),
    "IntOGen": load_intogen_genes("../data/IntOGen_drivers.tsv"),
    "Bailey et al.": load_bailey_genes("../data/Bailey2018_drivers.tsv"),
}

driver_source_support = Counter()
for source_name, genes in reference_sources.items():
    normalized_genes = {str(gene).upper() for gene in genes}
    driver_source_support.update(normalized_genes)
    print(f"[{source_name}] {len(normalized_genes)} genes")

# Any candidate with driver evidence remains out of the negative class.
driver_exclusion_set = set(driver_source_support)
# Positives require independent support from at least two of the four references.
driver_genes = all_genes & {
    gene for gene, support in driver_source_support.items()
    if support >= MIN_DRIVER_SOURCE_SUPPORT
}
# A pathway graph can only represent genes in the Reactome mapping. Keep the
# positive class graph-ready, while retaining every reference gene in the
# exclusion set so unmapped candidates never become negatives.
pathway_mapping_for_labels = pd.read_csv(
    "../data/processed/reactome_human_gene_pathway_mapping.tsv",
    sep="\t",
    dtype=str,
)
reactome_mapped_genes = set(
    pathway_mapping_for_labels.iloc[:, 1:].stack().dropna().str.upper()
)
unmapped_driver_genes = driver_genes - reactome_mapped_genes
driver_genes &= reactome_mapped_genes
print(f"[DRIVER EXCLUSION SET] {len(driver_exclusion_set)} unique genes across all sources")
print(
    f"[DRIVERS] {len(driver_genes)} protein-coding genes labeled positive "
    f"(supported by >= {MIN_DRIVER_SOURCE_SUPPORT} references and Reactome-mapped)"
)
print(f"[DRIVERS] {len(unmapped_driver_genes)} supported drivers retained as unlabeled (no Reactome mapping)")

non_drivers = all_genes - (driver_exclusion_set & all_genes)
print(
    f"[NON-DRIVERS] {len(non_drivers)} raw candidates "
    "(excluded from NCG/CGC/IntOGen/Bailey)"
)


[NCG] 3347 genes
[CGC] 763 genes
[IntOGen] 633 genes
[Bailey et al.] 299 genes
[DRIVER EXCLUSION SET] 3519 unique genes across all sources
[DRIVERS] 687 protein-coding genes labeled positive (supported by >= 2 references and Reactome-mapped)
[DRIVERS] 149 supported drivers retained as unlabeled (no Reactome mapping)
[NON-DRIVERS] 16606 raw candidates (excluded from NCG/CGC/IntOGen/Bailey)


In [4]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nChecking files:")
for path in [
    "../data/NCG_cancergenes.tsv",
    "../data/IntOGen_drivers.tsv",
    "../data/Bailey2018_drivers.tsv",
]:
    p = Path(path).resolve()
    print(f"{path}")
    print(f"  -> {p}")
    print(f"  Exists: {p.exists()}")

Current working directory:
/Users/ericsali/Documents/2024_Winter/Project_gnn/reactome_markers/gnn_pathways/ASPIRE-GNN/process/tcga-driver-gene-pipeline

Checking files:
../data/NCG_cancergenes.tsv
  -> /Users/ericsali/Documents/2024_Winter/Project_gnn/reactome_markers/gnn_pathways/ASPIRE-GNN/process/data/NCG_cancergenes.tsv
  Exists: True
../data/IntOGen_drivers.tsv
  -> /Users/ericsali/Documents/2024_Winter/Project_gnn/reactome_markers/gnn_pathways/ASPIRE-GNN/process/data/IntOGen_drivers.tsv
  Exists: True
../data/Bailey2018_drivers.tsv
  -> /Users/ericsali/Documents/2024_Winter/Project_gnn/reactome_markers/gnn_pathways/ASPIRE-GNN/process/data/Bailey2018_drivers.tsv
  Exists: True


In [5]:
config.NCG_FILE

PosixPath('../data/NCG_cancergenes.tsv')

## 3. Mutation-frequency filter

Excludes any candidate with mutation frequency >= `config.MUTATION_FREQUENCY_THRESHOLD` in any
TCGA cancer type. Skipped entirely if `config.USE_MUTATION_FILTER` is `False`.


In [6]:
MAX_MUTATION_FREQUENCY = 0.031
pathway_mapping_file = Path("../data/processed/reactome_human_gene_pathway_mapping.tsv")

pathway_mapping = pd.read_csv(pathway_mapping_file, sep="\t", dtype=str)
mapped_pathway_genes = set(
    pathway_mapping.iloc[:, 1:].stack().dropna().str.upper()
)
print(f"[PATHWAY MAPPING] {len(mapped_pathway_genes)} genes map to graph pathways")

before = len(non_drivers)
non_drivers = {
    gene.upper() for gene in non_drivers
    if gene.upper() in mapped_pathway_genes
}
print(
    f"[PATHWAY MAPPING] {len(non_drivers)} eligible candidates "
    f"(-{before - len(non_drivers)} unmapped)"
)

before = len(non_drivers)
non_drivers = apply_mutation_frequency_filter(
    non_drivers,
    config.MUTATION_FREQUENCY_FILE,
    threshold=MAX_MUTATION_FREQUENCY,
)
print(
    f"[MUTATION FILTER] {len(non_drivers)} remaining at "
    f"< {MAX_MUTATION_FREQUENCY:.0%} (-{before - len(non_drivers)})"
)


[PATHWAY MAPPING] 11932 genes map to graph pathways
[PATHWAY MAPPING] 8864 eligible candidates (-7742 unmapped)
[MUTATION FILTER] 3371 remaining at < 3% (-5493)


## 4. Disease-pathway filter

Excludes candidates belonging to a Reactome pathway that is a descendant of the top-level
*Disease* pathway. Skipped entirely if `config.USE_PATHWAY_FILTER` is `False`.


In [7]:
if config.USE_PATHWAY_FILTER:
    pathway_genes = build_disease_pathway_genes(
        "../../data/reactome/ReactomePathways.gmt",
        "../../data/reactome/reactome_relations.csv",
    )
    pd.DataFrame(sorted(pathway_genes), columns=["gene"]).to_csv(
        config.DISEASE_PATHWAY_GENES_FILE, index=False
    )
    print(f"[PATHWAY FILTER] {len(pathway_genes)} disease-pathway genes")

    before = len(non_drivers)
    non_drivers = apply_pathway_filter(non_drivers, pathway_genes)
    print(
        f"[PATHWAY FILTER] {len(non_drivers)} remaining "
        f"(-{before - len(non_drivers)})"
    )
else:
    print("[PATHWAY FILTER] skipped (config.USE_PATHWAY_FILTER is False)")


[PATHWAY FILTER] kept 782 disease-related pathways, skipped 2048 unrelated pathways
[PATHWAY FILTER] 2505 disease-pathway genes
[PATHWAY FILTER] 2806 remaining (-565)


## 5. Save four separate files

`driver_genes.txt`, `non_driver_genes.txt`, and `unlabeled_genes.txt` each hold one gene set
on its own (one gene per line); `gene_labels.csv` combines drivers + non-drivers into a
single labeled table for anything downstream that wants the pair together (unlabeled genes
are intentionally left out of that file since they have no `label` value).


In [8]:
# Keep every gene that remains after the reference, mapping, mutation, and pathway filters.
non_drivers = set(non_drivers)
print(f"[NEGATIVE SELECTION] Keeping all {len(non_drivers)} eligible pathway-mapped negatives")

# Every protein-coding gene outside the supervised positive/negative sets is unlabeled.
unlabeled_genes = all_genes - driver_genes - non_drivers
print(f"[UNLABELED] {len(unlabeled_genes)} genes excluded from both classes")

driver_path = config.PROCESSED_DIR / "driver_genes.txt"
nondriver_path = config.PROCESSED_DIR / "non_driver_genes.txt"
unlabeled_path = config.PROCESSED_DIR / "unlabeled_genes.txt"

driver_path.write_text("\n".join(sorted(driver_genes)) + "\n")
nondriver_path.write_text("\n".join(sorted(non_drivers)) + "\n")
unlabeled_path.write_text("\n".join(sorted(unlabeled_genes)) + "\n")

labels_df = build_gene_labels(driver_genes, non_drivers)
print(labels_df["label"].value_counts())
labels_df.to_csv(config.GENE_LABELS_FILE, index=False)

print(f"[DONE] Drivers ({len(driver_genes)}): {driver_path}")
print(f"[DONE] Non-drivers ({len(non_drivers)}, pathway-mapped): {nondriver_path}")
print(f"[DONE] Unlabeled ({len(unlabeled_genes)}): {unlabeled_path}")
print(f"[DONE] Combined labels (drivers + non-drivers only): {config.GENE_LABELS_FILE}")

labels_df.head()


[NEGATIVE SELECTION] Keeping all 2806 eligible pathway-mapped negatives
[UNLABELED] 16577 genes excluded from both classes
label
0    2806
1     687
Name: count, dtype: int64
[DONE] Drivers (687): ../data/processed/driver_genes.txt
[DONE] Non-drivers (2806, pathway-mapped): ../data/processed/non_driver_genes.txt
[DONE] Unlabeled (16577): ../data/processed/unlabeled_genes.txt
[DONE] Combined labels (drivers + non-drivers only): ../data/processed/gene_labels_driver_vs_nondriver.csv


,gene,label
0,FOXA2,1
1,NUP133,1
2,SMARCA4,1
3,PBRM1,1
4,TRIM24,1
